# L13 · 网格策略（Grid Trading）

**预计时长**：75 min | **难度**：⭐⭐⭐⭐ | **前置**：L12

## 本节目标
1. 网格逻辑：震荡市的「低买高卖」自动化
2. 向量化实现：cumsum 累加穿越次数 → 连续仓位
3. 网格 vs 双均线 vs 海龟的适用场景对比
4. qtrader.GridStrategy 源码精读

## 第 1 段：金融概念

### 网格策略核心思想
在一个价格区间内**等分若干网格**，每上穿一条线加 1 仓，每下穿一条线减 1 仓：
```
价格轴（lookback 区间 low~high 等分 10 格）：
━━━━━━━━━━━━━━ high（ref）
─ ─ ─ ─ ─ ─ 网格 9
─ ─ ─ ─ ─ ─ 网格 8
...
─ ─ ─ ─ ─ ─ 网格 1
━━━━━━━━━━━━━━ low（ref）
```
- **震荡市**：价格在区间内来回穿越，反复吃到 0.5%-1% 的小利润
- **趋势上涨**：仓位逐渐加满 → 变成「满仓踏空上涨」
- **趋势下跌**：仓位逐渐减到 0 → 至少不深套

### 与双均线/海龟的对比
| 维度 | 双均线 / 海龟 | 网格 |
|------|--------------|------|
| 市场偏好 | 趋势市 | 震荡市 |
| 信号触发 | 突破 | 穿越网格线 |
| 仓位形态 | 0/1 二值（或 ATR 缩放） | 0 ~ max_pos 连续 |
| 失效场景 | 频繁假突破 | 单边趋势（踏空或深套） |

### A 股适配
- 适合**高波动但不深跌**的标的（ETF、蓝筹、可转债）
- 不适合**单边趋势**（乐视网式崩盘）
- 网格数 grid_n 通常 5-20，太少粗糙，太多交易成本吃光利润

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation（共享 _data/_style）+ project root
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (
    _cwd / 'learning' / 'phase1_foundation'
    if (_cwd / 'learning' / 'phase1_foundation' / '_data.py').exists()
    else _cwd.parent / 'phase1_foundation'
)
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data
from qtrader import (
    BacktestEngine, CostModel,
    DualMAStrategy, TurtleStrategy, GridStrategy,
    print_comparison_table,
)

## 第 2 段：手写网格信号（向量化核心）

In [ ]:
byd = get_stock_data('002594').set_index('date')

def grid_signal(close: pd.Series, grid_n: int = 10, lookback: int = 60) -> pd.Series:
    \"\"\"手写网格信号（向量化，无 K 线级 for 循环）。

    关键技巧：
    - 网格线 = ref_low + i × step，i ∈ [1, grid_n]
    - 每日检查「是否上穿第 i 条线」+1，「是否下穿第 i 条线」-1
    - 累加 → 连续仓位
    \"\"\"
    ref_high = close.rolling(lookback).max().shift(1)
    ref_low = close.rolling(lookback).min().shift(1)
    step = (ref_high - ref_low) / grid_n
    prev_close = close.shift(1)

    position_change = pd.Series(0.0, index=close.index)
    for i in range(1, grid_n + 1):
        # 「网格线循环」是可接受例外（非 K 线循环）
        grid_line = ref_low + i * step
        cross_up = (prev_close < grid_line) & (close >= grid_line)
        cross_down = (prev_close > grid_line) & (close <= grid_line)
        position_change += cross_up.astype(float) - cross_down.astype(float)

    raw = position_change.cumsum().clip(0, grid_n) / grid_n
    return raw.shift(1).fillna(0)

sig = grid_signal(byd['close'], grid_n=10, lookback=60)
print(f"平均仓位: {sig.mean():.2%}")
print(f"最大仓位: {sig.max():.2%}")
print(f"调仓次数: {(sig.diff().abs() > 1e-6).sum()}")

## 第 3 段：回测 + 对比 qtrader.GridStrategy

In [ ]:
engine = BacktestEngine()
result = engine.run(byd.reset_index(), GridStrategy(grid_n=10, lookback=60))

fig, ax = plt.subplots(figsize=(13, 5))
result.benchmark_nav.plot(ax=ax, label='买入持有', alpha=0.5)
result.nav.plot(ax=ax, label='qtrader.GridStrategy', linewidth=1.5)
ax.set_title('比亚迪 网格策略 vs 买入持有')
ax.legend()
plt.tight_layout(); plt.show()

print(f"网格策略夏普: {result.metrics['sharpe']:.3f}")
print(f"买入持有夏普:  (未计算，参考 nav.plot)")
print(f"网格持仓时长: {result.metrics['exposure_time']:.2%}")

## 第 4 段：网格参数敏感性

- `grid_n`（网格数）：影响交易频率与单次利润
- `lookback`（参考窗口）：影响区间宽度

In [ ]:
from qtrader.optimize import grid_search

# 注意：grid_n 越大不一定越好，因为交易成本累积
top = grid_search(engine, byd.reset_index(), GridStrategy,
                  {'grid_n': [5, 10, 20], 'lookback': [30, 60, 120]},
                  metric='sharpe', top_n=5)
print("Top 5 网格参数组合：")
print(top[['grid_n', 'lookback', 'sharpe', 'n_trades', 'exposure_time']].to_string(index=False))

## 第 5 段：何时用网格（实战指南）

✅ 适合：
- ETF（沪深 300、中证 500、科创 50）：长期震荡向上
- 高股息蓝筹（银行、公用事业）：股价「磨」
- 可转债（下有债底、上有赎回）：天然区间

❌ 不适合：
- 强趋势股（次新股、题材股）：要么踏空要么深套
- 流动性差的股票：网格触发后无法成交
- 涨跌停频繁的股票：一字板时网格失效

### 🔮 下节 L14：多股选股
三个策略都是「择时」（何时买卖），下节转向「选股」（买什么）。